In [ ]:
import pandas as pd
import os
import re

# --- CONFIGURATION ---
root_dir = os.getcwd()
index_file = 'Redox_Index.csv'
output_file = 'Master_Redox.csv'

# 1. Load Index
index_df = pd.read_csv(index_file)
index_dict = index_df.set_index('Source_File').to_dict('index')

all_data = []

# 2. Iterate through folders
for loop_folder in os.listdir(root_dir):
    folder_path = os.path.join(root_dir, loop_folder)
    if not os.path.isdir(folder_path): 
        continue
    
    print(f"Processing folder: {loop_folder}")
    run_groups = {}
    
    # --- STEP A: Identify and Group Files (using Suffix-Stripping for long names) ---
    for filename in os.listdir(folder_path):
        lookup_name = f"{loop_folder}\\{filename}" 
        
        if lookup_name in index_dict:
            # Create Run Key by stripping '_RE...Max.csv'
            run_key = re.sub(r'_RE\d*Max\.csv$', '', filename, flags=re.IGNORECASE)
            
            if run_key not in run_groups: 
                run_groups[run_key] = []
            run_groups[run_key].append((filename, lookup_name))

    # --- STEP B: Process each Group for Pt1/Pt2 ---
    for run_key, file_pairs in run_groups.items():
        file_pairs.sort(key=lambda x: x[0]) 
        
        for i, (filename, lookup_name) in enumerate(file_pairs):
            file_path = os.path.join(folder_path, filename)
            meta = index_dict[lookup_name]
            
            try:
                df = pd.read_csv(file_path, delimiter=';')
                
                # Determine Label (Pt1, Pt2, or Pt)
                label = 'Pt' if len(file_pairs) == 1 else f'Pt{i+1}'
                
                # Identify data column
                max_cols = [c for c in df.columns if 'Max' in c]
                if not max_cols: continue
                redox_col = max_cols[0]
                
                # Map metadata and data
                df['LOOPNr'] = loop_folder
                df['ID'] = meta['ID']
                df['Grounding'] = meta['Grounding']
                df['Electrode_Label'] = label
                df['Redox_mV'] = df[redox_col]
                
                # NEW: The field UpDn is already in the file, so we just keep it
                # We use a selection that includes 'UpDn'
                cols_to_keep = [
                    'LOOPNr', 'ID', 'Depth', 'Redox_mV', 
                    'Electrode_Label', 'Grounding', 'DateTime', 'UpDn'
                ]
                
                # Ensure all requested columns exist in this specific file
                existing_cols = [c for c in cols_to_keep if c in df.columns]
                df = df[existing_cols]
                
                all_data.append(df)
            except Exception as e:
                print(f"Error processing {filename}: {e}")

# 3. Final Merge and Multi-Level Sort
if all_data:
    master_redox = pd.concat(all_data, ignore_index=True)

    # Chronological sorting safety
    master_redox['DateTime_Sort'] = pd.to_datetime(master_redox['DateTime'], errors='coerce')

    # Final Sorting Hierarchy
    master_redox = master_redox.sort_values(by=['LOOPNr', 'DateTime_Sort', 'ID', 'Depth'])

    # Final Clean up
    master_redox = master_redox.drop(columns=['DateTime_Sort']).reset_index(drop=True)
    
    # Save to CSV
    master_redox.to_csv(output_file, index=False)
    
    print(f"--- SUCCESS! ---")
    print(f"Master file created at: {output_file}")
    print(f"Final columns: {list(master_redox.columns)}")
else:
    print("CRITICAL: No data found. Check your index and folder contents.")

Processing folder: DEMO
Processing folder: LOOP2
Processing folder: LOOP3
Processing folder: LOOP4
Processing folder: LOOP6
--- SUCCESS! ---
Master file created at: Master_Redox.csv
Final columns: ['LOOPNr', 'ID', 'Depth', 'Redox_mV', 'Electrode_Label', 'Grounding', 'DateTime', 'UpDn']


C:\Users\ivany\AppData\Local\Temp\ipykernel_38544\2159673615.py:80: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  master_redox = pd.concat(all_data, ignore_index=True)


: 